## Data Extraction

In [1]:
# import libraries

from datetime import datetime, timedelta, timezone

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
import os
from dotenv import load_dotenv
import pandas as pd

### 15 Minute Stock Data

We'll retrieve 5 & 15 Minute price data for TSLA, NVDA, PLTR, COIN, & OKLO

These 5 stocks have heavy options activity and I primarily focus on trading these tickers

In [10]:
load_dotenv()

ALPACA_API_KEY = os.getenv("ALPACA_API_KEY")
ALPACA_SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")

if not ALPACA_API_KEY or not ALPACA_SECRET_KEY:
    raise ValueError("Missing Alpaca keys. Check your .env file.")

client = StockHistoricalDataClient(ALPACA_API_KEY, ALPACA_SECRET_KEY)

# pull 2 years of TSLA data 15M price data
end = datetime.now(timezone.utc) - timedelta(minutes=20)
start = end - timedelta(days=365 * 2) 

all_dfs = []

current_start = start

while current_start < end:
    current_end = min(current_start + timedelta(days=90), end)

    req = StockBarsRequest(
        symbol_or_symbols="COIN",
        timeframe=TimeFrame(5, TimeFrameUnit.Minute),
        start=current_start,
        end=current_end,
        limit=10000,
    )

    bars = client.get_stock_bars(req).df

    if not bars.empty:
        all_dfs.append(bars)

    print(f"Pulled {current_start.date()} → {current_end.date()} : {len(bars)} rows")

    current_start = current_end

df = pd.concat(all_dfs).reset_index()

print("Total rows:", len(df))
df.head()

Pulled 2024-02-28 → 2024-05-28 : 10000 rows
Pulled 2024-05-28 → 2024-08-26 : 10000 rows
Pulled 2024-08-26 → 2024-11-24 : 10000 rows
Pulled 2024-11-24 → 2025-02-22 : 10000 rows
Pulled 2025-02-22 → 2025-05-23 : 10000 rows
Pulled 2025-05-23 → 2025-08-21 : 10000 rows
Pulled 2025-08-21 → 2025-11-19 : 10000 rows
Pulled 2025-11-19 → 2026-02-17 : 10000 rows
Pulled 2026-02-17 → 2026-02-27 : 1506 rows
Total rows: 81506


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap
0,COIN,2024-02-28 17:00:00+00:00,210.435,211.2000,208.38,211.2000,434465.0,5272.0,209.858263
1,COIN,2024-02-28 17:05:00+00:00,211.166,211.6300,210.38,210.8750,238866.0,4011.0,211.133156
2,COIN,2024-02-28 17:10:00+00:00,210.770,211.1950,209.25,211.0379,258458.0,3356.0,210.383626
3,COIN,2024-02-28 17:15:00+00:00,211.040,211.4599,207.80,207.8900,296406.0,4428.0,209.852641
4,COIN,2024-02-28 17:20:00+00:00,207.960,209.0000,206.50,206.7100,334472.0,4715.0,207.499718


In [11]:
df.to_csv("COIN_5min_2yr.csv", index=False)
print("Saved COIN_5min_2yr.csv")

Saved COIN_5min_2yr.csv
